# Popolazione per Comune per Anno (2002–2025)

Due formati di fonte:
- **2002–2018**: `PopolazioneEta-SingolaArea-Comune_XXXXXX-Nome.csv` — un file per comune, anni in colonna
- **2019–2025**: `POSAS_<anno>_it_Comuni.csv` — un file per anno, comuni in riga

Output: griglia con comuni come righe, anni come colonne, popolazione totale come valori.

In [ ]:
import re
import pandas as pd
from pathlib import Path

DATA_DIR = Path('/content')

In [ ]:
def parse_posas_file(filepath):
    """
    Legge un file POSAS_<anno>_it_Comuni.csv.
    Salta le prime 2 righe (titolo + intestazione), usa indici di colonna:
      col 0 = codice comune
      col 1 = nome comune
      col 2 = età
      col -1 = totale popolazione
    """
    df = pd.read_csv(
        filepath,
        sep=';',
        skiprows=2,
        header=None,
        dtype=str,
    )
    return pd.DataFrame({
        'codice':  df.iloc[:, 0],
        'comune':  df.iloc[:, 1],
        'eta':     pd.to_numeric(df.iloc[:, 2], errors='coerce'),
        'totale':  pd.to_numeric(df.iloc[:, -1], errors='coerce'),
    }).dropna(subset=['eta', 'totale'])

In [ ]:
# --- Carica file POSAS (2019–2025) ---
ANNI_POSAS = list(range(2019, 2026))
risultati_posas = {}

for anno in ANNI_POSAS:
    filepath = DATA_DIR / f'POSAS_{anno}_it_Comuni.csv'
    if not filepath.exists():
        print(f'File non trovato: {filepath.name}')
        continue
    df = parse_posas_file(filepath)
    totali = df[df['eta'] == 999][['codice', 'comune', 'totale']].copy()
    totali = totali.set_index(['codice', 'comune'])['totale']
    risultati_posas[anno] = totali
    print(f'{anno}: {len(totali)} comuni')

griglia_posas = pd.DataFrame(risultati_posas)
griglia_posas.index.names = ['codice', 'comune']
griglia_posas.columns.name = 'anno'
print(f'\nGriglia POSAS: {griglia_posas.shape}')

In [ ]:
# --- Verifica coerenza 2019 tra le due fonti ---
# Allinea gli indici sul codice comune (le due fonti potrebbero avere
# formati leggermente diversi per il codice: 001001 vs 028001 ecc.)

storica_2019 = griglia_storica[2019].rename('storica')
posas_2019   = griglia_posas[2019].rename('posas') if 2019 in griglia_posas.columns else None

if posas_2019 is not None:
    # Confronta sui comuni presenti in entrambe le fonti (join sul codice)
    storica_idx = storica_2019.reset_index(level='comune')
    posas_idx   = posas_2019.reset_index(level='comune')

    confronto = storica_idx[['storica']].join(posas_idx[['posas']], how='inner')
    confronto['diff'] = confronto['posas'] - confronto['storica']
    confronto['diff_pct'] = (confronto['diff'] / confronto['storica'] * 100).round(2)

    n_comuni  = len(confronto)
    n_uguali  = (confronto['diff'] == 0).sum()
    n_diversi = (confronto['diff'] != 0).sum()
    diff_max  = confronto['diff'].abs().max()

    print(f'Comuni confrontabili: {n_comuni}')
    print(f'  Valori identici:    {n_uguali}')
    print(f'  Valori diversi:     {n_diversi}')
    print(f'  Differenza massima: {diff_max}')

    if n_diversi > 0:
        print('\nPrimi 10 comuni con differenza:')
        print(confronto[confronto['diff'] != 0].head(10))
else:
    print('POSAS 2019 non caricato — impossibile confrontare')

In [ ]:
# --- Combina le due griglie ---
# Storica: 2002-2018  |  POSAS: 2019-2025
# Il 2019 viene preso da POSAS (fonte più recente); quello storico era solo per verifica.

griglia_storica_no2019 = griglia_storica.drop(columns=[2019])
griglia = pd.concat([griglia_storica_no2019, griglia_posas], axis=1).sort_index(axis=1)
griglia.index.names = ['codice', 'comune']
griglia.columns.name = 'anno'

print(f'Griglia combinata: {griglia.shape[0]} comuni x {griglia.shape[1]} anni')
print(f'Anni coperti: {griglia.columns.min()} – {griglia.columns.max()}')
griglia.head(5)

In [ ]:
ANNI = list(range(2019, 2026))  # 2019-2025
DATA_DIR = Path('/content')

risultati = {}

for anno in ANNI:
    filepath = DATA_DIR / f'POSAS_{anno}_it_Comuni.csv'
    if not filepath.exists():
        print(f'File non trovato: {filepath}')
        continue
    df = parse_posas_file(filepath)
    totali = df[df['eta'] == 999][['codice', 'comune', 'totale']].copy()
    totali = totali.set_index(['codice', 'comune'])['totale']
    risultati[anno] = totali
    print(f'{anno}: {len(totali)} comuni caricati')

print(f'\nAnni caricati: {list(risultati.keys())}')

In [ ]:
# Costruisci la griglia: comuni x anni
griglia = pd.DataFrame(risultati)
griglia.index.names = ['codice', 'comune']
griglia.columns.name = 'anno'

print(f'Griglia: {griglia.shape[0]} comuni x {griglia.shape[1]} anni')
griglia.head(10)

In [ ]:
# Verifica: valori mancanti (comuni presenti solo in alcuni anni)
mancanti = griglia.isnull().sum()
if mancanti.sum() > 0:
    print('Valori mancanti per anno:')
    print(mancanti[mancanti > 0])
else:
    print('Nessun valore mancante')

In [ ]:
# Salva il risultato in CSV
output_path = DATA_DIR / 'popolazione_comuni_per_anno.csv'
griglia.to_csv(output_path)
print(f'Salvato in: {output_path}')

In [ ]:
# Esempio: cerca un comune specifico
comune_cerca = 'Abano Terme'
mask = griglia.index.get_level_values('comune') == comune_cerca
if mask.any():
    print(f'Popolazione di {comune_cerca} per anno:')
    print(griglia[mask].T)
else:
    print(f'Comune "{comune_cerca}" non trovato')